# Task 1 — Recipe Grid Search (Colab T4)
Chạy thử nghiệm 6 recipe YAML trên 1,000 dòng dữ liệu `xlam_2k` (1 epoch, seed 42) để so sánh 5 Arms với Anchor.

In [ ]:
# --- Cell 1: Gán Token & Clone repo private về Colab GPU ------------------------------
import os, getpass, subprocess, sys
from pathlib import Path

REPO_HTTPS = "github.com/thanhhao98/ChatSystem.git"
MARKER = "docs/contracts/cli.md"

token = os.environ.get("GITHUB_TOKEN")
if not token and not Path(MARKER).exists() and not Path(f"/content/ChatSystem/{MARKER}").exists():
    token = getpass.getpass("Dán GITHUB_TOKEN của bạn vào đây (chữ sẽ bị ẩn): ").strip()
    os.environ["GITHUB_TOKEN"] = token

if not Path(MARKER).exists():
    if Path(f"/content/ChatSystem/{MARKER}").exists():
        %cd /content/ChatSystem
    elif token:
        print("Đang clone repo private lên ổ đĩa Colab GPU...")
        !git clone --quiet https://{token}@{REPO_HTTPS} /content/ChatSystem
        %cd /content/ChatSystem
    else:
        raise RuntimeError("Thiếu GITHUB_TOKEN để clone repo private!")

REPO = Path.cwd()
print(f"Thành công! Thư mục làm việc trên Colab GPU: {REPO}")

In [ ]:
# --- Cell 2: Cài đặt thư viện & Mount Google Drive --------------------------------------
%cd {REPO}

!pip install -q -r requirements-train.txt
!nvidia-smi

OUT_BASE = "runs"
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_BASE = "/content/drive/MyDrive/ChatSystem/runs"
    os.makedirs(OUT_BASE, exist_ok=True)
    print(f"Đã mount Google Drive thành công! Thư mục lưu runs: {OUT_BASE}")
except Exception as e:
    print(f"Lưu ý: Không mount được Drive ({e}). Sử dụng đĩa VM mặc định: {OUT_BASE}")

In [ ]:
# --- Cell 3: Anchor Run (public_anchor.yaml) ----------------------------------------------
%cd {REPO}
out_dir = f"{OUT_BASE}/public_anchor_s42"

!python training/finetune_qlora.py --recipe training/recipes/public_anchor.yaml \
    --data data/public/xlam_2k.train.jsonl --val data/public/xlam_2k.val.jsonl \
    --output-dir {out_dir} --no-amp

!python training/predict_toolcall.py --backend transformers \
    --model Qwen/Qwen2.5-0.5B-Instruct --adapter {out_dir} \
    --eval data/public/xlam_2k.eval.json --out results/public/anchor_pred.jsonl

!python training/eval_toolcall.py --gold data/public/xlam_2k.eval.json \
    --pred results/public/anchor_pred.jsonl --out results/public/anchor_results.json

In [ ]:
# --- Cell 4: Run 5 Arms -------------------------------------------------------------------
%cd {REPO}
arms_info = [
    ("recipe_arm_r8", "Qwen/Qwen2.5-0.5B-Instruct"),
    ("recipe_arm_lr2e4", "Qwen/Qwen2.5-0.5B-Instruct"),
    ("recipe_arm_comp0", "Qwen/Qwen2.5-0.5B-Instruct"),
    ("recipe_arm_1b5", "Qwen/Qwen2.5-1.5B-Instruct"),
    ("recipe_arm_free", "Qwen/Qwen2.5-0.5B-Instruct")
]

for arm, model_name in arms_info:
    print(f"=== Running {arm} ({model_name}) ===")
    out_dir = f"{OUT_BASE}/{arm}_s42"
    
    !python training/finetune_qlora.py --recipe training/recipes/{arm}.yaml \
        --data data/public/xlam_2k.train.jsonl --val data/public/xlam_2k.val.jsonl \
        --output-dir {out_dir} --no-amp
        
    !python training/predict_toolcall.py --backend transformers \
        --model {model_name} --adapter {out_dir} \
        --eval data/public/xlam_2k.eval.json --out results/public/{arm}_pred.jsonl
        
    !python training/eval_toolcall.py --gold data/public/xlam_2k.eval.json \
        --pred results/public/{arm}_pred.jsonl --out results/public/{arm}_results.json

In [ ]:
# --- Cell 5: Bootstrap CI & McNemar Test (Xuất ra Markdown) -------------------------------
%cd {REPO}

!python training/bootstrap_ci.py \
    --run anchor=results/public/anchor_results.json \
    --run arm_r8=results/public/recipe_arm_r8_results.json \
    --run arm_lr2e4=results/public/recipe_arm_lr2e4_results.json \
    --run arm_comp0=results/public/recipe_arm_comp0_results.json \
    --run arm_1b5=results/public/recipe_arm_1b5_results.json \
    --run arm_free=results/public/recipe_arm_free_results.json \
    --pair anchor arm_r8 \
    --pair anchor arm_lr2e4 \
    --pair anchor arm_comp0 \
    --pair anchor arm_1b5 \
    --pair anchor arm_free \
    --md results/public/recipe_grid_bootstrap.md

# In nội dung bảng báo cáo kết quả ra màn hình
!cat results/public/recipe_grid_bootstrap.md